## quick merge to ttl

just merges stuff:
1. load ontology jsonld
2. load match jsonld files
3. write one ttl

same prefix style as `Bundesliga23_24_test.ttl`

In [ ]:
# setup
import json
from pathlib import Path

from rdflib import Graph, Namespace

project_root = Path(".").resolve()
folder_with_match_files = project_root / "jsonld_batch_output"
ontology_file = project_root / "ontology.jsonld"

Ontology: True | Match folder: True


In [ ]:
# merge + export ttl

CONTEXT = {
    "@vocab": "https://w3id.org/football-cdf/core#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
}


def build_kg_ttl(ontology_path: Path, jsonld_folder: Path, output_path: Path, max_matches=None) -> Graph:
    with open(ontology_path, "r", encoding="utf-8") as f:
        ontology_data = json.load(f)
    ontology_nodes = ontology_data if isinstance(ontology_data, list) else ontology_data.get("@graph", [ontology_data])
    all_jsonld = sorted(jsonld_folder.glob("*.jsonld"))
    files_to_use = all_jsonld[:max_matches] if max_matches else all_jsonld
    graph_nodes = list(ontology_nodes)
    for filepath in files_to_use:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        nodes = data.get("@graph", [data]) if isinstance(data, dict) else data
        graph_nodes.extend(nodes)

    merged = {"@context": CONTEXT, "@graph": graph_nodes}
    g = Graph()
    g.parse(data=json.dumps(merged, ensure_ascii=False), format="json-ld")

    # keep prefixes same as existing ttl
    FCDF = Namespace("https://w3id.org/football-cdf/core#")
    g.bind("", FCDF)  # @prefix : <...#>
    g.bind("dcterms", Namespace("http://purl.org/dc/terms/"))
    g.bind("owl", Namespace("http://www.w3.org/2002/07/owl#"))
    g.bind("rdfs", Namespace("http://www.w3.org/2000/01/rdf-schema#"))
    g.bind("vann", Namespace("http://purl.org/vocab/vann/"))
    g.bind("xsd", Namespace("http://www.w3.org/2001/XMLSchema#"))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    g.serialize(destination=str(output_path), format="turtle")
    return g

### quick test

writes `Bundesliga23_24_test.ttl` with 1 match

In [ ]:
# Test: 1 match only
test_output = project_root / "Bundesliga23_24_test.ttl"
build_kg_ttl(
    ontology_path=ontology_file,
    jsonld_folder=folder_with_match_files,
    output_path=test_output,
    max_matches=1,
)

Building TEST knowledge graph (1 match)...
  Loaded ontology: 86 nodes
  Using 1 match files
  Total nodes: 3876 (ontology + match data)
  Saved: C:\Users\dyury\Desktop\Master Thesis\Bundesliga23_24_test.ttl
Done. You can import this TTL into GraphDB.
